In [9]:
import os
import google.generativeai as genai
from tavily import TavilyClient
from dotenv import load_dotenv

load_dotenv()

GOOGLE_API_KEY = os.getenv('GEMINI_API_KEY')
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')

d:\Proyectos\alianza-f1-knowledge-agent\Cursos\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\tcpad\AppData\Local\Temp\ipykernel_15936\3315354158.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [10]:
import os
import re
from tavily import TavilyClient

api_key = os.environ.get("TAVILY_API_KEY")
if not api_key:
    raise ValueError("la clave TAVILY_API_KEY no fue encontrada.")

cliente_tavily = TavilyClient(api_key=api_key)

ciudad = "Tulum"
tavily_query = f"restaurantes en {ciudad} tripadvisor con mayor cantidad de reviews y rango de precios"

print("Iniciando la búsqueda agéntica por URLs de Tripadvisor con Tavily...")
tripadvisor_url = None

try:
    tavily_results = cliente_tavily.search(query=tavily_query, max_results=5)

    if tavily_results and tavily_results["results"]:
        print(f"Tavily encontró {len(tavily_results['results'])} resultados. Analizando...")
        for result in tavily_results["results"]:
            url = result["url"]
            if "tripadvisor.com" in url or "tripadvisor.com.mx" in url:
                tripadvisor_url = url
                break
    if not tripadvisor_url:
        print("Ninguna URL relevante de Tripadvisor fue encontrada en los primeros resultados.")
    else:
        print("Tavily no encontró resultados para la búsqueda agéntica.")
except Exception as e:
    print(f"Error en la búsqueda agéntica con Tavily: {e}. Verifica tu clave de API o tu conexión.")

if tripadvisor_url:
    clean_url = re.sub(r'-oa\d+-', '-', tripadvisor_url)
    tripadvisor_url = clean_url
    print(f"✅ URL encontrada limpia de paginación.")

print("*" * 50)
print(f"URL Final de Tripadvisor para el raspado: {tripadvisor_url if tripadvisor_url else 'NO ENCONTRADO'}")
print("*" * 50)

Iniciando la búsqueda agéntica por URLs de Tripadvisor con Tavily...
Tavily encontró 5 resultados. Analizando...
Tavily no encontró resultados para la búsqueda agéntica.
✅ URL encontrada limpia de paginación.
**************************************************
URL Final de Tripadvisor para el raspado: https://www.tripadvisor.com/Restaurants-g150813-zfp10955-Tulum_Yucatan_Peninsula.html
**************************************************


In [11]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
from selenium.common.exceptions import WebDriverException, TimeoutException

def scrape_restaurantes_info(url):
    if not url:
        print("Error: URL vacia o no localizada para el raspado.")
        return None
    
    try:
        service = Service(ChromeDriverManager().install())
        options = webdriver.ChromeOptions()
        options.add_argument("--headless")
        options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36")
        
        driver = webdriver.Chrome(service=service, options=options)
        driver.set_page_load_timeout(30)
        
    except Exception as e:
        print(f"Error al inicializar el driver de Selenium: {e}")
        return None

    try:
        print(f"Tratando cargar la página con Selenium: {url}")
        driver.get(url)
        
        driver.implicitly_wait(10)
        
        response_text = driver.page_source
    except TimeoutException:
        print(f"Error de limite de tiempo al cargar la página ({url}).")
        return None
    except WebDriverException as e:
        print(f"Error al cargar la página {url} con Selenium: {e}. Puede ser debido a un bloqueo o a que la URL no es válida.")
        return None
    finally:
        driver.quit()
        
    soup = BeautifulSoup(response_text, 'html.parser')
    return soup

In [12]:
soup_tripadvisor = None

if 'tripadvisor_url' in locals() and tripadvisor_url:
    print(f"\nTratando raspar la página identificada: {tripadvisor_url}")
    soup_tripadvisor = scrape_restaurantes_info(tripadvisor_url)
    
    if soup_tripadvisor:
        print("HTML de la página de Tripadvisor obtenido con éxito!")
        page_title_tag = soup_tripadvisor.find('title')
        if page_title_tag:
            print(f"Título de la página: {page_title_tag.get_text(strip=True)}")
        else:
            print("No fue posible encontrar el título de la página.")
    else:
        print("Falla al raspar la página de Tripadvisor. Verifica la URL o si la página está bloqueada.")
else:
    print("No existe URL de Tripadvisor válida para el raspado (obtenido en el bloque 1).")

print("*" * 50)


Tratando raspar la página identificada: https://www.tripadvisor.com/Restaurants-g150813-zfp10955-Tulum_Yucatan_Peninsula.html
Error al inicializar el driver de Selenium: [WinError 4551] An Application Control policy has blocked this file
Falla al raspar la página de Tripadvisor. Verifica la URL o si la página está bloqueada.
**************************************************
